# Tutorial 5 — Continuous, Network & Domain Models

Five applied models: a **logistic** stock-and-flow, **epidemic spread on two network topologies**, a predator–prey **phase portrait**, **SIR flatten-the-curve**, and a **3-node supply chain**.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
%matplotlib inline

from sim_lab.core import (
    Stock,
    Flow,
    SystemDynamicsSimulation,
    NetworkSimulation,
    create_small_world_network,
    create_scale_free_network,
    PredatorPreySimulation,
    EpidemiologicalSimulation,
    Factory,
    Distributor,
    Retailer,
    SupplyChainLink,
    SupplyChainSimulation,
    base_stock_policy,
    constant_demand,
)

random_seed = 42
np.random.seed(random_seed)
random.seed(random_seed)
print(f"Reproducibility seed locked: {random_seed}")

## 1. System Dynamics — logistic limits to growth

We build $\dot{N} = rN(1 - N/K)$ from a `Stock` (population) fed by one `Flow` whose rate closes the feedback loop through the stock's own value. The Verhulst solution rises in an S-curve and **levels off at the carrying capacity** $K$.

In [ ]:
K, r = 1000.0, 0.5

def logistic_growth(state, time):
    N = state['Population']
    return r * N * (1.0 - N / K)

# Flow name 'flow_from_growth_to_Population' routes the rate into the Population
# stock; 'growth' is not itself a stock, so nothing is decremented.
stocks = {'Population': Stock('Population', 10.0)}
flows = {'flow_from_growth_to_Population': Flow('flow_from_growth_to_Population',
                                                logistic_growth)}
sd = SystemDynamicsSimulation(stocks=stocks, flows=flows,
                             days=200, dt=0.1, random_seed=random_seed)
P = sd.run_simulation()['stock_Population']
t = np.linspace(0, 200, len(P))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t, P, label='population')
ax.axhline(K, color='crimson', ls='--', label=f'carrying capacity K={K:.0f}')
ax.set_xlabel('time'); ax.set_ylabel('population')
ax.set_title('Logistic growth levels off at the carrying capacity')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

assert P[-1] > 0.95 * K, 'population must approach K'
assert all(P[i + 1] >= P[i] - 1e-6 for i in range(len(P) - 1)), 'logistic growth is monotone'
print(f'population levels at {P[-1]:.1f}  (K = {K:.0f}, {100*P[-1]/K:.1f}% of capacity)')

## 2. Network — epidemic spread on two topologies

An SI process: each step, infected nodes infect each susceptible neighbour with probability $\beta$. We seed one node and compare a **small-world** graph (locally clustered) against a **scale-free** graph (a few high-degree hubs). Hubs make scale-free networks burn through the population far faster.

In [ ]:
def run_epidemic(net, days=50, beta=0.12, seed_node=0):
    np.random.seed(random_seed); random.seed(random_seed)
    for n in net.nodes.values():
        n.update_attribute('state', 'susceptible')
    net.nodes[seed_node].update_attribute('state', 'infected')
    infected = [1]

    def spread(network, day):
        newly = set()
        for nid, node in network.nodes.items():
            if node.attributes.get('state') == 'infected':
                for nb in node.neighbors:
                    if (network.nodes[nb].attributes.get('state') == 'susceptible'
                            and random.random() < beta):
                        newly.add(nb)
        for nid in newly:
            network.nodes[nid].update_attribute('state', 'infected')
        infected.append(sum(1 for n in network.nodes.values()
                            if n.attributes.get('state') == 'infected'))

    net.update_function = spread
    net.days = days
    net.run_simulation()
    return infected

N = 300
np.random.seed(random_seed); random.seed(random_seed)
sw = create_small_world_network(num_nodes=N, k=4, beta=0.1)
sf = create_scale_free_network(num_nodes=N, m=2)
sw_inf = run_epidemic(sw)
sf_inf = run_epidemic(sf)

def time_to_half(arr):
    for i, v in enumerate(arr):
        if v >= N / 2:
            return i
    return len(arr)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sw_inf, label=f'small-world (t_50={time_to_half(sw_inf)})')
ax.plot(sf_inf, label=f'scale-free (t_50={time_to_half(sf_inf)})')
ax.set_xlabel('step'); ax.set_ylabel('infected nodes')
ax.set_title('Epidemic spread: scale-free topology is faster')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

assert time_to_half(sf_inf) < time_to_half(sw_inf), 'scale-free must spread faster'
print(f'scale-free reaches 50% by step {time_to_half(sf_inf)}, '
      f'small-world by step {time_to_half(sw_inf)}.')

## 3. Predator–Prey — phase portrait

The Lotka–Volterra equations produce coupled oscillations: prey boom, then predators boom on the abundant prey, then predators over-consume and crash, then prey recover. Plotted in the **phase plane** (prey vs predator) this traces a closed orbit — the signature of a neutrally stable cycle.

In [ ]:
pp = PredatorPreySimulation(
    initial_prey=40.0, initial_predators=9.0,
    prey_growth_rate=1.0, predation_rate=0.1,
    predator_death_rate=0.5, predator_growth_factor=0.02,
    days=300, dt=0.005, random_seed=random_seed,
)
res = pp.run_simulation()
prey, predators = np.array(res['prey']), np.array(res['predators'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(prey, label='prey'); axes[0].plot(predators, label='predators')
axes[0].set_xlabel('day'); axes[0].set_ylabel('population')
axes[0].set_title('Predator–prey time series'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(prey, predators, color='purple')
axes[1].scatter([prey[0]], [predators[0]], color='green', zorder=5, label='start')
axes[1].set_xlabel('prey'); axes[1].set_ylabel('predators')
axes[1].set_title('Phase portrait (closed orbit)'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

assert prey.min() > 0 and predators.min() > 0, 'neither population should go extinct'
assert predators.max() > 10 and prey.max() > 30, 'clear oscillations must occur'
print(f'prey range {prey.min():.1f}-{prey.max():.1f}; predator range {predators.min():.1f}-{predators.max():.1f}')

## 4. Epidemiological — flatten the curve

The SIR model with transmission rate $\beta$ and recovery rate $\gamma$. The reproduction number is $R_0 = \beta/\gamma$. Cutting $\beta$ (distancing, masks) lowers $R_0$ and **flattens the infection curve**: a smaller, later peak that keeps demand under healthcare capacity.

In [ ]:
def run_sir(beta, days=200):
    e = EpidemiologicalSimulation(
        population_size=10000, initial_infected=10,
        beta=beta, gamma=0.1, days=days, random_seed=random_seed,
    )
    e.run_simulation()
    return e

e_hi = run_sir(0.5)   # R0 = 5
e_lo = run_sir(0.2)   # R0 = 2
compartments_hi = e_hi.get_compartments()
compartments_lo = e_lo.get_compartments()
days = np.arange(len(compartments_hi['infected']))
peak_day_hi, peak_hi = e_hi.get_peak_infection()
peak_day_lo, peak_lo = e_lo.get_peak_infection()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(days, compartments_hi['infected'], color='crimson',
        label=f'beta=0.5 (R0={e_hi.get_reproduction_number():.1f}), peak={peak_hi:.0f}')
ax.plot(days, compartments_lo['infected'], color='steelblue',
        label=f'beta=0.2 (R0={e_lo.get_reproduction_number():.1f}), peak={peak_lo:.0f}')
ax.axhline(peak_hi, color='crimson', ls=':', alpha=0.5)
ax.set_xlabel('day'); ax.set_ylabel('infected')
ax.set_title('Flatten the curve: lower beta -> smaller, later peak')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

assert peak_lo < peak_hi, 'lower beta must reduce the peak'
print(f'peak infection: beta=0.5 -> {peak_hi:.0f} (day {peak_day_hi}); '
      f'beta=0.2 -> {peak_lo:.0f} (day {peak_day_lo})')

## 5. Supply Chain — a 3-node network

A **Factory** produces, a **Distributor** ships downstream, and a **Retailer** serves customer demand under a constant demand stream. Each echelon follows a base-stock ordering policy. We read the retailer's inventory drawdown, the factory's production response to the demand signal, and the aggregate service level and profit.

In [ ]:
factory = Factory(name='Factory', production_capacity=60.0, production_cost=2.0,
                 initial_inventory=120.0, capacity=2000.0, lead_time=1)
distributor = Distributor(name='Distributor', shipping_cost=0.4,
                         initial_inventory=90.0, capacity=2000.0, lead_time=2)
retailer = Retailer(name='Retailer', selling_price=8.0, holding_cost=0.05,
                    stockout_cost=2.0, initial_inventory=3500.0,
                    capacity=6000.0, lead_time=1)
links = [SupplyChainLink(factory, distributor),
         SupplyChainLink(distributor, retailer)]
nodes = {'Factory': factory, 'Distributor': distributor, 'Retailer': retailer}
policies = {
    'Factory': base_stock_policy(200.0),
    'Distributor': base_stock_policy(90.0),
    'Retailer': base_stock_policy(3500.0),
}
sc = SupplyChainSimulation(
    nodes=nodes, links=links,
    demand_generator=constant_demand(30.0),
    ordering_policies=policies, days=100, random_seed=random_seed,
)
result = sc.run_simulation()
metrics = result['overall_metrics']
days = np.arange(len(result['Retailer']['inventory']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(days, result['Retailer']['inventory'], label='retailer inventory')
axes[0].plot(days, result['Distributor']['inventory'], label='distributor inventory')
axes[0].set_xlabel('day'); axes[0].set_ylabel('inventory')
axes[0].set_title('Inventory drawdown across echelons'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(days, result['Retailer']['demand'], color='gray', label='customer demand')
axes[1].plot(days, result['Factory']['production'], color='crimson', label='factory production')
axes[1].set_xlabel('day'); axes[1].set_ylabel('units / day')
axes[1].set_title('Demand signal reaches production'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

service = metrics['service_level'][0]
profit = metrics['total_profit'][0]
print(f'service level = {service:.3f}')
print(f'total profit  = {profit:,.0f}')
print(f'retailer served {sum(result["Retailer"]["sales"]):.0f} of '
      f'{sum(result["Retailer"]["demand"]):.0f} units demanded')
assert service >= 0.99, 'well-stocked retailer should meet all demand'

## Validation & interpretation

| Model | Law / behaviour | Check |
|---|---|---|
| System Dynamics | logistic levels at $K$ | population reaches $>99\%$ of $K$ ✅ |
| Network | scale-free spreads faster than small-world | $t_{50}(\text{SF}) < t_{50}(\text{SW})$ ✅ |
| Predator–Prey | closed phase orbit, no extinction | both populations stay $>0$, clear oscillation ✅ |
| Epidemiological | lower $\beta$ lowers the peak | peak(0.2) $<$ peak(0.5) ✅ |
| Supply Chain | 3-node chain runs, demand is served | service level $\approx 1.0$ ✅ |

Two big ideas recur. First, **structure determines dynamics**: the same SI rule spreads slowly on a clustered small-world graph but explosively on a hub-driven scale-free graph, and the same SIR equations give a gentle vs. overwhelming epidemic depending only on $\beta$. Second, **feedback sets the steady state**: the logistic's self-limiting term parks the population at $K$, while the predator–prey feedback (more prey $\to$ more predators $\to$ fewer prey) sustains a permanent oscillation. The supply chain closes the loop too — the customer demand signal propagates upstream and drives factory production.